# 🔬 1-Bit Digital Comparator — Physics Practical Analysis
**Experiment No. 01 | Tri-Chandra Multiple Campus, Tribhuvan University**

---

This notebook contains the complete data analysis, error estimation, statistical summary, and graphical analysis for the **1-Bit Digital Comparator** experiment.  
The comparator was built using ICs: **7408 (AND)**, **7404 (NOT)**, **7402 (NOR)**, and a **BC547 transistor** for output indication.

---

## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Instrument and circuit parameters
POWER_SUPPLY_V  = 3.0    # V — actual supply used in practical
VCC_IC          = 5.0    # V — IC rated supply
VOLTMETER_RANGE = 5.0    # V (full scale = 10 divisions)
LEAST_COUNT     = 0.5    # V per division (instrument uncertainty)
HIGH_VOLTAGE    = 2.35   # V — measured HIGH output
LOW_VOLTAGE     = 0.0    # V — measured LOW output
UNCERTAINTY     = 0.5    # V — one division (least count)

print('Configuration loaded.')
print(f'Power supply  : {POWER_SUPPLY_V} V')
print(f'Least count   : {LEAST_COUNT} V/division')
print(f'Measured HIGH : {HIGH_VOLTAGE} V ± {UNCERTAINTY} V')

## 2. Experimental Data

Data recorded from the breadboard circuit for all 4 input combinations (A, B) ∈ {00, 01, 10, 11}.  
Output voltages were measured using a voltmeter (range: 5V, least count: 0.5 V/division).

In [ ]:
# Load processed data
df = pd.read_csv('../data/processed_data.csv')
print('Processed Data:')
display(df)

## 3. Truth Table (Theoretical vs Experimental)

In [ ]:
truth_table = pd.DataFrame({
    'A': [0, 0, 1, 1],
    'B': [0, 1, 0, 1],
    'A=B (theory)': [1, 0, 0, 1],
    'A>B (theory)': [0, 0, 1, 0],
    'A<B (theory)': [0, 1, 0, 0],
    'V_AeqB (V)':   [2.35, 0.0, 0.0, 2.35],
    'V_AgtB (V)':   [0.0, 0.0, 2.35, 0.0],
    'V_AltB (V)':   [0.0, 2.35, 0.0, 0.0],
})

# Verify experimental vs theory
def to_logic(v, threshold=1.0):
    return 1 if v >= threshold else 0

truth_table['A=B (exp)'] = truth_table['V_AeqB (V)'].apply(to_logic)
truth_table['A>B (exp)'] = truth_table['V_AgtB (V)'].apply(to_logic)
truth_table['A<B (exp)'] = truth_table['V_AltB (V)'].apply(to_logic)

truth_table['Match?'] = (
    (truth_table['A=B (theory)'] == truth_table['A=B (exp)']) &
    (truth_table['A>B (theory)'] == truth_table['A>B (exp)']) &
    (truth_table['A<B (theory)'] == truth_table['A<B (exp)'])
).map({True: '✅ Match', False: '❌ Mismatch'})

display(truth_table[['A','B','A=B (theory)','A=B (exp)',
                      'A>B (theory)','A>B (exp)',
                      'A<B (theory)','A<B (exp)','Match?']])

matches = (truth_table['Match?'] == '✅ Match').sum()
print(f'\nAccuracy: {matches}/4 input combinations matched theory → {matches/4*100:.1f}%')

## 4. Error Analysis

### 4.1 Instrument Error (Voltmeter)

The voltmeter has a full-scale range of 5 V across 10 divisions:

$$\text{Least Count} = \frac{\text{Range}}{\text{Total Divisions}} = \frac{5\,\text{V}}{10} = 0.5\,\text{V/division}$$

So the uncertainty in every voltage reading: $\delta V = \pm 0.5\,\text{V}$

### 4.2 Percentage Deviation from Supply Voltage

The power supply was set to $V_{supply} = 3.0\,\text{V}$.  
The measured HIGH output was $V_{HIGH} = 2.35\,\text{V}$.

$$\text{Absolute error} = |V_{supply} - V_{HIGH}| = |3.0 - 2.35| = 0.65\,\text{V}$$

$$\text{Relative error} = \frac{0.65}{3.0} = 0.217$$

$$\text{Percentage error} = 21.7\%$$

In [ ]:
# Error calculations
measured_high   = HIGH_VOLTAGE
reference       = POWER_SUPPLY_V

abs_error = abs(reference - measured_high)
rel_error = abs_error / reference
pct_error = rel_error * 100

print('─' * 45)
print('  Error Analysis — HIGH Output Voltage')
print('─' * 45)
print(f'  Measured HIGH voltage  : {measured_high:.2f} V')
print(f'  Reference (Vcc)        : {reference:.2f} V')
print(f'  Instrument uncertainty : ±{UNCERTAINTY:.2f} V')
print()
print(f'  Absolute error  = {abs_error:.2f} V')
print(f'  Relative error  = {rel_error:.4f}')
print(f'  Percentage error= {pct_error:.2f} %')
print()
print('  Physical reason: TTL ICs have a typical')
print('  output HIGH voltage of ~2.4V (Voh_min),')
print('  which is consistent with 2.35V ± 0.5V.')
print('─' * 45)

## 5. Statistical Analysis

We take the HIGH output voltage measurements (where output should be HIGH) to perform statistical analysis.

In [ ]:
# All HIGH output readings from the experiment
# (A=B: rows 0,3; A>B: row 2; A<B: row 1)
high_measurements = np.array([2.35, 2.35, 2.35, 2.35])  # All HIGH outputs
low_measurements  = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])  # All LOW outputs

n    = len(high_measurements)
mean = np.mean(high_measurements)
var  = np.var(high_measurements, ddof=1)
std  = np.std(high_measurements, ddof=1)
se   = std / np.sqrt(n)  # Standard error
prob_err = 0.6745 * std  # Probable error

print('Statistical Analysis — HIGH Output Readings')
print('─' * 40)
print(f'  N (measurements)  = {n}')
print(f'  Mean (x̄)         = {mean:.4f} V')
print(f'  Variance (s²)     = {var:.4f} V²')
print(f'  Std Deviation (s) = {std:.4f} V')
print(f'  Std Error (SE)    = {se:.4f} V')
print(f'  Probable Error    = {prob_err:.4f} V')
print()
print(f'  Final result: V_HIGH = {mean:.2f} ± {se:.4f} V (SE)')
print(f'                V_HIGH = {mean:.2f} ± {UNCERTAINTY:.2f} V (instrument)')
print('─' * 40)
print()
print('Note: Zero variance in HIGH readings confirms')
print('consistent IC behavior across all test cases.')

## 6. Graphical Analysis

In [ ]:
# Graph 1 — Output Voltages with Error Bars
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('1-Bit Digital Comparator — Measured Output Voltages',
             fontsize=14, fontweight='bold')

input_labels = ['A=0,B=0', 'A=0,B=1', 'A=1,B=0', 'A=1,B=1']
colors       = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
outputs      = ['V_AeqB (V)', 'V_AgtB (V)', 'V_AltB (V)']
titles       = ['Output: A = B', 'Output: A > B', 'Output: A < B']

for ax, col, title in zip(axes, outputs, titles):
    bars = ax.bar(input_labels, truth_table[col], color=colors,
                  edgecolor='black', linewidth=0.8, width=0.55)
    ax.errorbar(input_labels, truth_table[col],
                yerr=[UNCERTAINTY]*4, fmt='none', color='black',
                capsize=6, linewidth=1.5)
    ax.axhline(y=HIGH_VOLTAGE, color='gray', linestyle='--',
               linewidth=1.2, alpha=0.7, label=f'HIGH = {HIGH_VOLTAGE} V')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Input (A, B)', fontsize=10)
    ax.set_ylabel('Voltage (V)', fontsize=10)
    ax.set_ylim(0, 3.5)
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, truth_table[col]):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    val + 0.05, f'{val:.2f}V',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/graph_1_output_voltages.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Graph 2 — Truth Table Heatmap
fig, ax = plt.subplots(figsize=(9, 5))

data_matrix = np.array([
    truth_table['A=B (theory)'].values,
    truth_table['A>B (theory)'].values,
    truth_table['A<B (theory)'].values,
], dtype=float)

im = ax.imshow(data_matrix, cmap='RdYlGn', aspect='auto',
               vmin=0, vmax=1, interpolation='nearest')
ax.set_xticks(range(4))
ax.set_xticklabels(['A=0,B=0', 'A=0,B=1', 'A=1,B=0', 'A=1,B=1'], fontsize=11)
ax.set_yticks(range(3))
ax.set_yticklabels(['A = B', 'A > B', 'A < B'], fontsize=12, fontweight='bold')
ax.set_title('Truth Table Heatmap\n(Green = HIGH / 1, Red = LOW / 0)',
             fontsize=13, fontweight='bold')

for i in range(3):
    for j in range(4):
        val = int(data_matrix[i, j])
        ax.text(j, i, 'HIGH\n(1)' if val else 'LOW\n(0)',
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='white' if val else 'black')

plt.colorbar(im, ax=ax, fraction=0.03).set_label('Logic Level')
plt.tight_layout()
plt.savefig('../figures/graph_2_truth_table_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 7. Logical Expressions Verification

The Boolean expressions derived from the truth table:

$$A > B = A \cdot \bar{B}$$

$$A < B = \bar{A} \cdot B$$

$$A = B = \bar{A}\cdot\bar{B} + A \cdot B = \overline{(A \oplus B)}$$


In [ ]:
# Verify Boolean expressions programmatically
print('Boolean Expression Verification')
print('─' * 50)
print(f'{"A":>4} {"B":>4} | {"A>B":>6} {"A<B":>6} {"A=B":>6} | {"Correct?":>10}')
print('─' * 50)

for A, B in [(0,0), (0,1), (1,0), (1,1)]:
    AgtB = A and not B
    AltB = not A and B
    AeqB = (not A and not B) or (A and B)
    
    exp_AgtB = truth_table.loc[(truth_table['A']==A) & (truth_table['B']==B), 'A>B (theory)'].values[0]
    exp_AeqB = truth_table.loc[(truth_table['A']==A) & (truth_table['B']==B), 'A=B (theory)'].values[0]
    exp_AltB = truth_table.loc[(truth_table['A']==A) & (truth_table['B']==B), 'A<B (theory)'].values[0]
    
    match = (int(AgtB)==exp_AgtB and int(AltB)==exp_AltB and int(AeqB)==exp_AeqB)
    print(f'{A:>4} {B:>4} | {int(AgtB):>6} {int(AltB):>6} {int(AeqB):>6} | {"✅ OK" if match else "❌ Fail":>10}')

print('─' * 50)
print('All Boolean expressions verified against truth table.')

## 8. Result Summary

| Quantity | Value |
|---|---|
| HIGH output voltage | 2.35 ± 0.50 V |
| LOW output voltage | 0.00 ± 0.50 V |
| Power supply used | 3.0 V |
| Percentage deviation (HIGH from Vcc) | 21.7% |
| Truth table accuracy | 4/4 (100%) |
| ICs used | 7408, 7404, 7402 |
| Transistor | BC547 |

In [ ]:
print('Final Result Summary')
print('='*45)
print(f'  V_HIGH = {HIGH_VOLTAGE:.2f} ± {UNCERTAINTY:.2f} V')
print(f'  V_LOW  = {LOW_VOLTAGE:.2f} ± {UNCERTAINTY:.2f} V')
print(f'  Truth table accuracy = 100% (4/4 combinations)')
print(f'  Percentage error (vs Vcc=3V) = 21.7%')
print('='*45)
print()
print('The 1-bit digital comparator was successfully')
print('designed and verified on a breadboard using')
print('TTL ICs (7408, 7404, 7402) and BC547 transistor.')
print('All output states matched theoretical predictions.')